# Introduction to PyTorch
PyTorch简介
你已经在本作业中编写了大量代码，实现了各种神经网络功能。Dropout、Batch Norm 和二维卷积是计算机视觉深度学习中的重要工具。你也努力让代码高效且向量化。
在作业的最后部分，我们将暂时离开你精心打造的代码库，转向两个流行的深度学习框架之一：本次我们选择 PyTorch。

## Why do we use deep learning frameworks?
我们为什么要使用深度学习框架？
* Our code will now run on GPUs! This will allow our models to train much faster. When using a framework like PyTorch you can harness the power of the GPU for your own custom neural network architectures without having to write CUDA code directly (which is beyond the scope of this class).
* 现在我们的代码可以在GPU上运行！这将让模型训练速度大大提升。使用像PyTorch这样的框架，你可以利用GPU的强大算力来实现自定义神经网络架构，而无需直接编写CUDA代码（这超出了本课程范围）。
* In this class, we want you to be ready to use one of these frameworks for your project so you can experiment more efficiently than if you were writing every feature you want to use by hand.
* 在本课程中，我们希望你能熟练使用这些框架进行项目开发，这样你可以更高效地进行实验，而不是手动实现每一个功能。
* We want you to stand on the shoulders of giants! PyTorch is an excellent frameworks that will make your lives a lot easier, and now that you understand their guts, you are free to use them :)
* 我们希望你能站在巨人的肩膀上！PyTorch是一个优秀的框架，会让你的工作变得更轻松。现在你已经理解了它的底层原理，可以自由地使用它们 :)
* Finally, we want you to be exposed to the sort of deep learning code you might run into in academia or industry.
* 最后，我们希望你能接触到学术界或工业界常见的深度学习代码。
## What is PyTorch?
什么是PyTorch？
PyTorch is a system for executing dynamic computational graphs over Tensor objects that behave similarly as numpy ndarray. It comes with a powerful automatic differentiation engine that removes the need for manual back-propagation.
PyTorch是一个可以在Tensor对象上执行动态计算图的系统，这些Tensor的行为类似于numpy的ndarray。它自带强大的自动求导引擎，无需手动实现反向传播。
## How do I learn PyTorch?
我该如何学习PyTorch？
One of our former instructors, Justin Johnson, made an excellent [tutorial](https://github.com/jcjohnson/pytorch-examples) for PyTorch.
我们的一位前任讲师 Justin Johnson 制作了一个很棒的 [PyTorch教程](https://github.com/jcjohnson/pytorch-examples)。
You can also find the detailed [API doc](http://pytorch.org/docs/stable/index.html) here. If you have other questions that are not addressed by the API docs, the [PyTorch forum](https://discuss.pytorch.org/) is a much better place to ask than StackOverflow.
你还可以在这里找到详细的 [API文档](http://pytorch.org/docs/stable/index.html)。如果你有API文档未解答的问题，[PyTorch论坛](https://discuss.pytorch.org/)比StackOverflow更适合提问。

# Table of Contents
目录
This assignment has 5 parts. You will learn PyTorch on **three different levels of abstraction**, which will help you understand it better and prepare you for the final project.
本作业共5部分。你将从**三个不同抽象层次**学习PyTorch，这有助于你更好地理解它并为最终项目做准备。
1. Part I, Preparation: we will use CIFAR-10 dataset.
1. 第一部分，准备：我们将使用CIFAR-10数据集。
2. Part II, Barebones PyTorch: **Abstraction level 1**, we will work directly with the lowest-level PyTorch Tensors.
2. 第二部分，基础PyTorch：**抽象层次1**，我们将直接操作底层PyTorch张量。
3. Part III, PyTorch Module API: **Abstraction level 2**, we will use `nn.Module` to define arbitrary neural network architecture.
3. 第三部分，PyTorch模块API：**抽象层次2**，我们将用`nn.Module`定义任意神经网络结构。
4. Part IV, PyTorch Sequential API: **Abstraction level 3**, we will use `nn.Sequential` to define a linear feed-forward network very conveniently.
4. 第四部分，PyTorch序列API：**抽象层次3**，我们将用`nn.Sequential`方便地定义线性前馈网络。
5. Part V, CIFAR-10 open-ended challenge: please implement your own network to get as high accuracy as possible on CIFAR-10. You can experiment with any layer, optimizer, hyperparameters or other advanced features.
5. 第五部分，CIFAR-10开放挑战：请实现你自己的网络，在CIFAR-10上尽可能获得高准确率。你可以尝试任何层、优化器、超参数或其他高级特性。
Here is a table of comparison:
下面是一个对比表：
| API           | Flexibility | Convenience |
|---------------|-------------|-------------|
| Barebone      | High        | Low         |
| `nn.Module`     | High        | Medium      |
| `nn.Sequential` | Low         | High        |


# GPU
GPU
You can manually switch to a GPU device on Colab by clicking `Runtime -> Change runtime type` and selecting `GPU` under `Hardware Accelerator`. You should do this before running the following cells to import packages, since the kernel gets restarted upon switching runtimes.
你可以在Colab中手动切换到GPU设备，方法是点击`Runtime -> Change runtime type`，然后在`Hardware Accelerator`下选择`GPU`。在运行下面的代码单元导入包之前，请先完成此操作，因为切换运行时会重启内核。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this tutorial.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)

using device: cuda


# Part I. Preparation

Now, let's load the CIFAR-10 dataset. This might take a couple minutes the first time you do it, but the files should stay cached after that.

In previous parts of the assignment we had to write our own code to download the CIFAR-10 dataset, preprocess it, and iterate through it in minibatches; PyTorch provides convenient tools to automate this process for us.

In [2]:
NUM_TRAIN = 49000

# The torchvision.transforms package provides tools for preprocessing data
# and for performing data augmentation; here we set up a transform to
# preprocess the data by subtracting the mean RGB value and dividing by the
# standard deviation of each RGB value; we've hardcoded the mean and std.
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])

# We set up a Dataset object for each split (train / val / test); Datasets load
# training examples one at a time, so we wrap each Dataset in a DataLoader which
# iterates through the Dataset and forms minibatches. We divide the CIFAR-10
# training set into train and val sets by passing a Sampler object to the
# DataLoader telling how it should sample from the underlying Dataset.
cifar10_train = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./cs231n/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

# Part II. Barebones PyTorch

PyTorch ships with high-level APIs to help us define model architectures conveniently, which we will cover in Part II of this tutorial. In this section, we will start with the barebone PyTorch elements to understand the autograd engine better. After this exercise, you will come to appreciate the high-level model API more.

We will start with a simple fully-connected ReLU network with two hidden layers and no biases for CIFAR classification.
This implementation computes the forward pass using operations on PyTorch Tensors, and uses PyTorch autograd to compute gradients. It is important that you understand every line, because you will write a harder version after the example.

When we create a PyTorch Tensor with `requires_grad=True`, then operations involving that Tensor will not just compute values; they will also build up a computational graph in the background, allowing us to easily backpropagate through the graph to compute gradients of some Tensors with respect to a downstream loss. Concretely if x is a Tensor with `x.requires_grad == True` then after backpropagation `x.grad` will be another Tensor holding the gradient of x with respect to the scalar loss at the end.


# 第二部分：基础 PyTorch

PyTorch 提供了高级 API，帮助我们方便地定义模型结构，这些内容将在本教程的第二部分介绍。在本节中，我们将从最基础的 PyTorch 元素开始，深入理解自动求导引擎。完成这个练习后，你会更加欣赏高级模型 API 的便利。

我们将从一个简单的全连接 ReLU 网络开始，这个网络有两个隐藏层，并且没有偏置项，用于 CIFAR 分类。
这个实现通过对 PyTorch 张量进行操作来计算前向传播，并利用 PyTorch 的自动求导功能来计算梯度。理解每一行代码都很重要，因为你之后会写一个更复杂的版本。

当我们创建一个 `requires_grad=True` 的 PyTorch 张量时，涉及该张量的操作不仅会计算数值，还会在后台构建计算图，使我们能够轻松地通过反向传播来计算某些张量相对于下游损失的梯度。具体来说，如果 x 是一个 `x.requires_grad == True` 的张量，那么在反向传播后，`x.grad` 会是另一个张量，保存 x 关于最终标量损失的梯度。

### PyTorch Tensors: Flatten Function
A PyTorch Tensor is conceptionally similar to a numpy array: it is an n-dimensional grid of numbers, and like numpy PyTorch provides many functions to efficiently operate on Tensors. As a simple example, we provide a `flatten` function below which reshapes image data for use in a fully-connected neural network.

Recall that image data is typically stored in a Tensor of shape N x C x H x W, where:

* N is the number of datapoints
* C is the number of channels
* H is the height of the intermediate feature map in pixels
* W is the height of the intermediate feature map in pixels

This is the right way to represent the data when we are doing something like a 2D convolution, that needs spatial understanding of where the intermediate features are relative to each other. When we use fully connected affine layers to process the image, however, we want each datapoint to be represented by a single vector -- it's no longer useful to segregate the different channels, rows, and columns of the data. So, we use a "flatten" operation to collapse the `C x H x W` values per representation into a single long vector. The flatten function below first reads in the N, C, H, and W values from a given batch of data, and then returns a "view" of that data. "View" is analogous to numpy's "reshape" method: it reshapes x's dimensions to be N x ??, where ?? is allowed to be anything (in this case, it will be C x H x W, but we don't need to specify that explicitly).



### PyTorch 张量：Flatten 函数

PyTorch 的张量在概念上类似于 numpy 数组：它是一个 n 维的数字网格，并且像 numpy 一样，PyTorch 提供了许多高效操作张量的函数。下面我们给出一个简单的例子——`flatten` 函数，用于将图像数据重新变形，以便用于全连接神经网络。

回忆一下，图像数据通常存储为形状为 N x C x H x W 的张量，其中：

* N 是数据点的数量
* C 是通道数
* H 是中间特征图的高度（像素）
* W 是中间特征图的宽度（像素）

这种表示方式适用于需要空间关系理解的操作，比如二维卷积，因为它能体现特征之间的空间位置关系。而当我们用全连接仿射层处理图像时，我们希望每个数据点都用一个单独的向量表示——此时分离不同通道、行和列已经没有意义。因此，我们使用“flatten”操作，将每个表示中的 `C x H x W` 数值合并成一个长向量。下面的 flatten 函数首先读取一批数据的 N、C、H、W，然后返回该数据的一个“视图”。“视图”类似于 numpy 的“reshape”方法：它将 x 的维度变形为 N x ??，其中 ?? 可以是任意值（在这里就是 C x H x W，但我们不需要显式指定）。

In [3]:
def flatten(x):
    N = x.shape[0] # read in N, C, H, W
    return x.view(N, -1)  # "flatten" the C * H * W values into a single vector per image

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()

Before flattening:  tensor([[[[ 0,  1],
          [ 2,  3],
          [ 4,  5]]],


        [[[ 6,  7],
          [ 8,  9],
          [10, 11]]]])
After flattening:  tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


### Barebones PyTorch: Two-Layer Network

Here we define a function `two_layer_fc` which performs the forward pass of a two-layer fully-connected ReLU network on a batch of image data. After defining the forward pass we check that it doesn't crash and that it produces outputs of the right shape by running zeros through the network.

You don't have to write any code here, but it's important that you read and understand the implementation.

In [4]:
import torch.nn.functional as F  # useful stateless functions

def two_layer_fc(x, params):
    """
    A fully-connected neural networks; the architecture is:
    NN is fully connected -> ReLU -> fully connected layer.
    Note that this function only defines the forward pass;
    PyTorch will take care of the backward pass for us.

    The input to the network will be a minibatch of data, of shape
    (N, d1, ..., dM) where d1 * ... * dM = D. The hidden layer will have H units,
    and the output layer will produce scores for C classes.

    Inputs:
    - x: A PyTorch Tensor of shape (N, d1, ..., dM) giving a minibatch of
      input data.
    - params: A list [w1, w2] of PyTorch Tensors giving weights for the network;
      w1 has shape (D, H) and w2 has shape (H, C).

    Returns:
    - scores: A PyTorch Tensor of shape (N, C) giving classification scores for
      the input data x.
    """
    # first we flatten the image
    x = flatten(x)  # shape: [batch_size, C x H x W]

    w1, w2 = params

    # Forward pass: compute predicted y using operations on Tensors. Since w1 and
    # w2 have requires_grad=True, operations involving these Tensors will cause
    # PyTorch to build a computational graph, allowing automatic computation of
    # gradients. Since we are no longer implementing the backward pass by hand we
    # don't need to keep references to intermediate values.
    # you can also use `.clamp(min=0)`, equivalent to F.relu()
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x


def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)  # minibatch size 64, feature dimension 50
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())  # you should see [64, 10]

two_layer_fc_test()

torch.Size([64, 10])


### Barebones PyTorch: Three-Layer ConvNet

Here you will complete the implementation of the function `three_layer_convnet`, which will perform the forward pass of a three-layer convolutional network. Like above, we can immediately test our implementation by passing zeros through the network. The network should have the following architecture:

1. A convolutional layer (with bias) with `channel_1` filters, each with shape `KW1 x KH1`, and zero-padding of two
2. ReLU nonlinearity
3. A convolutional layer (with bias) with `channel_2` filters, each with shape `KW2 x KH2`, and zero-padding of one
4. ReLU nonlinearity
5. Fully-connected layer with bias, producing scores for C classes.

Note that we have **no softmax activation** here after our fully-connected layer: this is because PyTorch's cross entropy loss performs a softmax activation for you, and by bundling that step in makes computation more efficient.

**HINT**: For convolutions: http://pytorch.org/docs/stable/nn.html#torch.nn.functional.conv2d; pay attention to the shapes of convolutional filters!
已压缩对话### Barebones PyTorch: Three-Layer ConvNet

在这里，你需要完成函数`three_layer_convnet`的实现，该函数将执行一个三层卷积网络的前向传播。和之前一样，我们可以通过将全零输入传入网络来立即测试你的实现。该网络的结构如下：

1. 一个卷积层（带偏置），包含`channel_1`个滤波器，每个滤波器的形状为`KW1 x KH1`，并且使用2像素的零填充（padding）
2. ReLU非线性激活
3. 一个卷积层（带偏置），包含`channel_2`个滤波器，每个滤波器的形状为`KW2 x KH2`，并且使用1像素的零填充
4. ReLU非线性激活
5. 一个带偏置的全连接层，输出C个类别的分数

注意：在全连接层之后**没有softmax激活**，这是因为PyTorch的交叉熵损失（cross entropy loss）会自动为你执行softmax操作，并且将这一步合并可以提升计算效率。

**提示**：关于卷积操作，可以参考：http://pytorch.org/docs/stable/nn.html#torch.nn.functional.conv2d；请注意卷积滤波器的形状！

In [5]:
def three_layer_convnet(x, params):
    """
    Performs the forward pass of a three-layer convolutional network with the
    architecture defined above.

    Inputs:
    - x: A PyTorch Tensor of shape (N, 3, H, W) giving a minibatch of images
    - params: A list of PyTorch Tensors giving the weights and biases for the
      network; should contain the following:
      - conv_w1: PyTorch Tensor of shape (channel_1, 3, KH1, KW1) giving weights
        for the first convolutional layer
      - conv_b1: PyTorch Tensor of shape (channel_1,) giving biases for the first
        convolutional layer
      - conv_w2: PyTorch Tensor of shape (channel_2, channel_1, KH2, KW2) giving
        weights for the second convolutional layer
      - conv_b2: PyTorch Tensor of shape (channel_2,) giving biases for the second
        convolutional layer
      - fc_w: PyTorch Tensor giving weights for the fully-connected layer. Can you
        figure out what the shape should be?
      - fc_b: PyTorch Tensor giving biases for the fully-connected layer. Can you
        figure out what the shape should be?

    Returns:
    - scores: PyTorch Tensor of shape (N, C) giving classification scores for x
    """
    """
    执行上述定义架构的三层卷积网络的前向传播。

    输入：
    - x：形状为（N，3，H，W）的PyTorch张量，提供一批图像
    - params：PyTorch张量的列表，提供网络的权重和偏置；应包含以下内容：
      - conv_w1：形状为（channel_1，3，KH1，KW1）的PyTorch张量，提供第一层卷积层的权重
      - conv_b1：形状为（channel_1，）的PyTorch张量，提供第一层卷积层的偏置
      - conv_w2：形状为（channel_2，channel_1，KH2，KW2）的PyTorch张量，提供第二层卷积层的权重
      - conv_b2：形状为（channel_2，）的PyTorch张量，提供第二层卷积层的偏置
      - fc_w：提供全连接层权重的PyTorch张量。你能弄清楚它的形状应该是什么吗？
      - fc_b：提供全连接层偏置的PyTorch张量。你能弄清楚它的形状应该是什么吗？

    返回：
    - scores：形状为（N，C）的PyTorch张量，提供x的分类分数
    """

    conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b = params
    scores = None
    
    ################################################################################
    # TODO: Implement the forward pass for the three-layer ConvNet.                #
    ################################################################################
    # 第一层卷积: Conv -> ReLU
    # 架构要求: padding=2
    x = F.conv2d(x, conv_w1, conv_b1, stride=1, padding=2)
    x = F.relu(x)
    
    # 第二层卷积: Conv -> ReLU
    # 架构要求: padding=1 (注意这里改了)
    x = F.conv2d(x, conv_w2, conv_b2, stride=1, padding=1)
    x = F.relu(x)
    
    # 展平: (N, C, H, W) -> (N, C*H*W)
    x = x.view(x.size(0), -1)
    
    # 全连接层
    x = x.mm(fc_w) + fc_b.view(1, -1)
    scores = x

    ################################################################################
    #                                 END OF YOUR CODE                             #
    ################################################################################
    return scores

After defining the forward pass of the ConvNet above, run the following cell to test your implementation.

When you run this function, scores should have shape (64, 10).

In [6]:
def three_layer_convnet_test():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, image size [3, 32, 32]

    conv_w1 = torch.zeros((6, 3, 5, 5), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b1 = torch.zeros((6,))  # out_channel
    conv_w2 = torch.zeros((9, 6, 3, 3), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b2 = torch.zeros((9,))  # out_channel

    # you must calculate the shape of the tensor after two conv layers, before the fully-connected layer
    fc_w = torch.zeros((9 * 32 * 32, 10))
    fc_b = torch.zeros(10)

    scores = three_layer_convnet(x, [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b])
    print(scores.size())  # you should see [64, 10]
three_layer_convnet_test()

torch.Size([64, 10])


### Barebones PyTorch: Initialization
Let's write a couple utility methods to initialize the weight matrices for our models.

- `random_weight(shape)` initializes a weight tensor with the Kaiming normalization method.
- `zero_weight(shape)` initializes a weight tensor with all zeros. Useful for instantiating bias parameters.

The `random_weight` function uses the Kaiming normal initialization method, described in:

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852
### Barebones PyTorch: Initialization

让我们编写几个用于初始化模型权重矩阵的工具函数。

- `random_weight(shape)` 用 Kaiming 标准化方法初始化权重张量。
- `zero_weight(shape)` 用全零初始化权重张量，适合用于偏置参数。

`random_weight`函数采用的是 Kaiming 正态初始化方法，相关介绍见：

He 等人，*Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*，ICCV 2015，https://arxiv.org/abs/1502.01852

In [7]:
def random_weight(shape):
    """
    Create random Tensors for weights; setting requires_grad=True means that we
    want to compute gradients for these Tensors during the backward pass.
    We use Kaiming normalization: sqrt(2 / fan_in)
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # conv weight [out_channel, in_channel, kH, kW]
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# create a weight of shape [3 x 5]
# you should see the type `torch.cuda.FloatTensor` if you use GPU.
# Otherwise it should be `torch.FloatTensor`
random_weight((3, 5))

tensor([[ 0.6297, -0.4228, -0.3756,  1.3675,  0.6327],
        [ 0.1341, -0.1721, -0.4153,  0.5979,  0.1876],
        [ 0.8832, -1.5357,  0.6337,  1.0343, -1.3023]], device='cuda:0',
       requires_grad=True)

### Barebones PyTorch: Check Accuracy
When training the model we will use the following function to check the accuracy of our model on the training or validation sets.

When checking accuracy we don't need to compute any gradients; as a result we don't need PyTorch to build a computational graph for us when we compute scores. To prevent a graph from being built we scope our computation under a `torch.no_grad()` context manager.
### Barebones PyTorch: Check Accuracy

在训练模型时，我们会使用下面的函数来检查模型在训练集或验证集上的准确率。

在检查准确率时，我们不需要计算任何梯度；因此，在计算分数时也不需要让 PyTorch 构建计算图。为了防止构建计算图，我们会把相关计算放在 `torch.no_grad()` 上下文管理器中。

In [8]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model_fn: A function that performs the forward pass of the model,
      with the signature scores = model_fn(x, params)
    - params: List of PyTorch Tensors giving parameters of the model

    Returns: Nothing, but prints the accuracy of the model
    """
    split = 'val' if loader.dataset.train else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

### BareBones PyTorch: Training Loop
We can now set up a basic training loop to train our network. We will train the model using stochastic gradient descent without momentum. We will use `torch.functional.cross_entropy` to compute the loss; you can [read about it here](http://pytorch.org/docs/stable/nn.html#cross-entropy).

The training loop takes as input the neural network function, a list of initialized parameters (`[w1, w2]` in our example), and learning rate.


### BareBones PyTorch：训练循环
现在我们可以设置一个基本的训练循环来训练我们的网络。我们将使用不带动量的随机梯度下降（SGD）来训练模型。我们会用 `torch.functional.cross_entropy` 来计算损失；你可以在[这里](http://pytorch.org/docs/stable/nn.html#cross-entropy)阅读相关内容。

训练循环的输入包括神经网络函数、初始化参数列表（在我们的例子中是 `[w1, w2]`），以及学习率。

In [9]:
def train_part2(model_fn, params, learning_rate):
    """
    Train a model on CIFAR-10.

    Inputs:
    - model_fn: A Python function that performs the forward pass of the model.
      It should have the signature scores = model_fn(x, params) where x is a
      PyTorch Tensor of image data, params is a list of PyTorch Tensors giving
      model weights, and scores is a PyTorch Tensor of shape (N, C) giving
      scores for the elements in x.
    - params: List of PyTorch Tensors giving weights for the model
    - learning_rate: Python scalar giving the learning rate to use for SGD

    Returns: Nothing
    """
    for t, (x, y) in enumerate(loader_train):
        # Move the data to the proper device (GPU or CPU)
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # Forward pass: compute scores and loss
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # Backward pass: PyTorch figures out which Tensors in the computational
        # graph has requires_grad=True and uses backpropagation to compute the
        # gradient of the loss with respect to these Tensors, and stores the
        # gradients in the .grad attribute of each Tensor.
        loss.backward()

        # Update parameters. We don't want to backpropagate through the
        # parameter updates, so we scope the updates under a torch.no_grad()
        # context manager to prevent a computational graph from being built.
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # Manually zero the gradients after running the backward pass
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()

### BareBones PyTorch: Train a Two-Layer Network
Now we are ready to run the training loop. We need to explicitly allocate tensors for the fully connected weights, `w1` and `w2`.

Each minibatch of CIFAR has 64 examples, so the tensor shape is `[64, 3, 32, 32]`.

After flattening, `x` shape should be `[64, 3 * 32 * 32]`. This will be the size of the first dimension of `w1`.
The second dimension of `w1` is the hidden layer size, which will also be the first dimension of `w2`.

Finally, the output of the network is a 10-dimensional vector that represents the probability distribution over 10 classes.

You don't need to tune any hyperparameters but you should see accuracies above 40% after training for one epoch.


### BareBones PyTorch：训练一个两层网络
现在我们已经可以运行训练循环了。我们需要为全连接权重 `w1` 和 `w2` 显式分配张量。

CIFAR 的每个小批量（minibatch）包含 64 个样本，因此张量的形状为 `[64, 3, 32, 32]`。

展平后，`x` 的形状应为 `[64, 3 * 32 * 32]`。这将作为 `w1` 的第一维大小。
`w1` 的第二维是隐藏层的大小，这也将作为 `w2` 的第一维。

最后，网络的输出是一个 10 维向量，表示对 10 个类别的概率分布。

你不需要调整任何超参数，但在训练一个 epoch 后，准确率应该能超过 40%。

In [10]:
hidden_layer_size = 4000
learning_rate = 1e-2

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)

Iteration 0, loss = 3.3644
Checking accuracy on the val set
Got 149 / 1000 correct (14.90%)

Iteration 100, loss = 2.0726
Checking accuracy on the val set
Got 352 / 1000 correct (35.20%)

Iteration 200, loss = 2.4171
Checking accuracy on the val set
Got 366 / 1000 correct (36.60%)

Iteration 300, loss = 2.1510
Checking accuracy on the val set
Got 366 / 1000 correct (36.60%)

Iteration 400, loss = 1.8926
Checking accuracy on the val set
Got 379 / 1000 correct (37.90%)

Iteration 500, loss = 1.9006
Checking accuracy on the val set
Got 421 / 1000 correct (42.10%)

Iteration 600, loss = 1.7828
Checking accuracy on the val set
Got 425 / 1000 correct (42.50%)

Iteration 700, loss = 2.0173
Checking accuracy on the val set
Got 424 / 1000 correct (42.40%)



### BareBones PyTorch: Training a ConvNet

In the below you should use the functions defined above to train a three-layer convolutional network on CIFAR. The network should have the following architecture:

1. Convolutional layer (with bias) with 32 5x5 filters, with zero-padding of 2
2. ReLU
3. Convolutional layer (with bias) with 16 3x3 filters, with zero-padding of 1
4. ReLU
5. Fully-connected layer (with bias) to compute scores for 10 classes

You should initialize your weight matrices using the `random_weight` function defined above, and you should initialize your bias vectors using the `zero_weight` function above.

You don't need to tune any hyperparameters, but if everything works correctly you should achieve an accuracy above 42% after one epoch.


### BareBones PyTorch：训练一个卷积神经网络

在下面的部分，你需要使用上面定义的函数，在 CIFAR 数据集上训练一个三层卷积神经网络。该网络的结构如下：

1. 卷积层（带偏置），使用 32 个 5x5 的卷积核，零填充为 2
2. ReLU 激活函数
3. 卷积层（带偏置），使用 16 个 3x3 的卷积核，零填充为 1
4. ReLU 激活函数
5. 全连接层（带偏置），用于计算 10 个类别的得分

你应该使用上面定义的 `random_weight` 函数来初始化权重矩阵，使用 `zero_weight` 函数来初始化偏置向量。

你不需要调整任何超参数，但如果一切正常，训练一个 epoch 后准确率应该能超过 42%。

In [13]:
learning_rate = 3e-3

channel_1 = 32
channel_2 = 16

conv_w1 = None
conv_b1 = None
conv_w2 = None
conv_b2 = None
fc_w = None
fc_b = None

################################################################################
# TODO: Initialize the parameters of a three-layer ConvNet.                    #
################################################################################


# 初始化参数
conv_w1 = random_weight((channel_1, 3, 5, 5))  # [out_channel, in_channel, kH, kW]
conv_b1 = zero_weight((channel_1,))              # 偏置用零初始化
conv_w2 = random_weight((channel_2, channel_1, 3, 3))
conv_b2 = zero_weight((channel_2,))
fc_w = random_weight((channel_2 * 32 * 32, 10))   # 修正形状：(in_features, out_features)
fc_b = zero_weight((10,))

# 整理参数列表并训练
params = [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b]
train_part2(three_layer_convnet, params, learning_rate)


################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

params = [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b]
train_part2(three_layer_convnet, params, learning_rate)

Iteration 0, loss = 2.8624
Checking accuracy on the val set
Got 161 / 1000 correct (16.10%)

Iteration 100, loss = 1.9750
Checking accuracy on the val set
Got 347 / 1000 correct (34.70%)

Iteration 200, loss = 1.7139
Checking accuracy on the val set
Got 377 / 1000 correct (37.70%)

Iteration 300, loss = 1.5365
Checking accuracy on the val set
Got 426 / 1000 correct (42.60%)

Iteration 400, loss = 1.6166
Checking accuracy on the val set
Got 454 / 1000 correct (45.40%)

Iteration 500, loss = 1.5082
Checking accuracy on the val set
Got 450 / 1000 correct (45.00%)

Iteration 600, loss = 1.8142
Checking accuracy on the val set
Got 478 / 1000 correct (47.80%)

Iteration 700, loss = 1.4102
Checking accuracy on the val set
Got 476 / 1000 correct (47.60%)

Iteration 0, loss = 1.3103
Checking accuracy on the val set
Got 494 / 1000 correct (49.40%)

Iteration 100, loss = 1.3216
Checking accuracy on the val set
Got 499 / 1000 correct (49.90%)

Iteration 200, loss = 1.2654
Checking accuracy on the 

# Part III. PyTorch Module API

Barebone PyTorch requires that we track all the parameter tensors by hand. This is fine for small networks with a few tensors, but it would be extremely inconvenient and error-prone to track tens or hundreds of tensors in larger networks.

PyTorch provides the `nn.Module` API for you to define arbitrary network architectures, while tracking every learnable parameters for you. In Part II, we implemented SGD ourselves. PyTorch also provides the `torch.optim` package that implements all the common optimizers, such as RMSProp, Adagrad, and Adam. It even supports approximate second-order methods like L-BFGS! You can refer to the [doc](http://pytorch.org/docs/master/optim.html) for the exact specifications of each optimizer.

To use the Module API, follow the steps below:

1. Subclass `nn.Module`. Give your network class an intuitive name like `TwoLayerFC`.

2. In the constructor `__init__()`, define all the layers you need as class attributes. Layer objects like `nn.Linear` and `nn.Conv2d` are themselves `nn.Module` subclasses and contain learnable parameters, so that you don't have to instantiate the raw tensors yourself. `nn.Module` will track these internal parameters for you. Refer to the [doc](http://pytorch.org/docs/master/nn.html) to learn more about the dozens of builtin layers. **Warning**: don't forget to call the `super().__init__()` first!

3. In the `forward()` method, define the *connectivity* of your network. You should use the attributes defined in `__init__` as function calls that take tensor as input and output the "transformed" tensor. Do *not* create any new layers with learnable parameters in `forward()`! All of them must be declared upfront in `__init__`.

After you define your Module subclass, you can instantiate it as an object and call it just like the NN forward function in part II.

### Module API: Two-Layer Network
Here is a concrete example of a 2-layer fully connected network:

以下是你提供内容的中文翻译：

# 第三部分：PyTorch Module API

Barebone PyTorch 需要我们手动管理所有参数张量。对于只有几个张量的小型网络来说，这还可以，但在更大的网络中需要管理几十甚至上百个张量，这将极其不方便且容易出错。

PyTorch 提供了 `nn.Module` API，允许你定义任意网络结构，并自动帮你追踪所有可学习参数。在第二部分中，我们自己实现了 SGD。PyTorch 还提供了 `torch.optim` 包，里面实现了所有常见的优化器，比如 RMSProp、Adagrad 和 Adam，甚至还支持近似二阶方法如 L-BFGS！你可以参考[官方文档](http://pytorch.org/docs/master/optim.html)了解每个优化器的具体用法。

要使用 Module API，请按照以下步骤操作：

1. 继承 `nn.Module`。给你的网络类起一个直观的名字，比如 `TwoLayerFC`。

2. 在构造函数 `__init__()` 中，将所有需要的层定义为类属性。像 `nn.Linear` 和 `nn.Conv2d` 这样的层对象本身也是 `nn.Module` 的子类，并且包含可学习参数，因此你无需自己实例化原始张量。`nn.Module` 会自动追踪这些内部参数。你可以参考[官方文档](http://pytorch.org/docs/master/nn.html)了解内置层的详细信息。**注意**：不要忘记先调用 `super().__init__()`！

3. 在 `forward()` 方法中，定义网络的*连接方式*。你应该将 `__init__` 中定义的属性作为函数调用，输入张量并输出“变换后”的张量。不要在 `forward()` 里新建带可学习参数的层！所有层都必须在 `__init__` 里提前声明。

定义好 Module 子类后，你可以像第二部分的前向函数一样实例化并调用它。

### Module API：两层全连接网络
下面是一个两层全连接网络的具体例子：

In [14]:
class TwoLayerFC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # assign layer objects to class attributes
        self.fc1 = nn.Linear(input_size, hidden_size)
        # nn.init package contains convenient initialization methods
        # http://pytorch.org/docs/master/nn.html#torch-nn-init
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        # forward always defines connectivity
        x = flatten(x)
        scores = self.fc2(F.relu(self.fc1(x)))
        return scores

def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)  # minibatch size 64, feature dimension 50
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())  # you should see [64, 10]
test_TwoLayerFC()

torch.Size([64, 10])


### Module API: Three-Layer ConvNet
It's your turn to implement a 3-layer ConvNet followed by a fully connected layer. The network architecture should be the same as in Part II:

1. Convolutional layer with `channel_1` 5x5 filters with zero-padding of 2
2. ReLU
3. Convolutional layer with `channel_2` 3x3 filters with zero-padding of 1
4. ReLU
5. Fully-connected layer to `num_classes` classes

You should initialize the weight matrices of the model using the Kaiming normal initialization method.

**HINT**: http://pytorch.org/docs/stable/nn.html#conv2d

After you implement the three-layer ConvNet, the `test_ThreeLayerConvNet` function will run your implementation; it should print `(64, 10)` for the shape of the output scores.
以下是你提供内容的中文翻译：

### Module API：三层卷积神经网络

现在轮到你来实现一个三层卷积神经网络，并在最后接一个全连接层。网络结构应与第二部分相同：

1. 卷积层，使用 `channel_1` 个 5x5 卷积核，零填充为 2
2. ReLU 激活函数
3. 卷积层，使用 `channel_2` 个 3x3 卷积核，零填充为 1
4. ReLU 激活函数
5. 全连接层，输出为 `num_classes` 个类别

你应该使用 Kaiming 正态初始化方法来初始化模型的权重矩阵。

**提示**: http://pytorch.org/docs/stable/nn.html#conv2d

当你实现好三层卷积神经网络后，`test_ThreeLayerConvNet` 函数会运行你的实现；它应该会输出分数的形状为 `(64, 10)`。

In [19]:

class ThreeLayerConvNet(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        ########################################################################
        # TODO: Set up the layers you need for a three-layer ConvNet with the  #
        # architecture defined above.                                          #
        ########################################################################
        self.fc1 = nn.Conv2d(in_channel, channel_1, kernel_size=5, stride=1, padding=2)
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Conv2d(channel_1, channel_2, kernel_size=3, stride=1, padding=1)
        nn.init.kaiming_normal_(self.fc2.weight)
        self.fc3 = nn.Linear(channel_2 * 32 * 32, num_classes)
        nn.init.kaiming_normal_(self.fc3.weight)

        ########################################################################
        #                          END OF YOUR CODE                            #
        ########################################################################

    def forward(self, x):
        scores = None
        ########################################################################
        # TODO: Implement the forward function for a 3-layer ConvNet. you      #
        # should use the layers you defined in __init__ and specify the        #
        # connectivity of those layers in forward()                            #
        ########################################################################
        
        # 1. 第一层：Conv -> ReLU
        x = F.relu(self.fc1(x))
        
        # 2. 第二层：Conv -> ReLU
        x = F.relu(self.fc2(x))
        
        # 3. 【关键】展平：把 (N, C, H, W) 变成 (N, C*H*W)
        x = flatten(x)
        
        # 4. 全连接层
        scores = self.fc3(x)

        ########################################################################
        #                             END OF YOUR CODE                         #
        ########################################################################
        return scores


def test_ThreeLayerConvNet():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, image size [3, 32, 32]
    model = ThreeLayerConvNet(in_channel=3, channel_1=12, channel_2=8, num_classes=10)
    scores = model(x)
    print(scores.size())  # you should see [64, 10]
test_ThreeLayerConvNet()

torch.Size([64, 10])


### Module API: Check Accuracy
Given the validation or test set, we can check the classification accuracy of a neural network.

This version is slightly different from the one in part II. You don't manually pass in the parameters anymore.
### Module API：检查准确率

给定验证集或测试集，我们可以检查神经网络的分类准确率。

这个版本与第二部分的实现略有不同。你不再需要手动传递参数。

In [20]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

### Module API: Training Loop
We also use a slightly different training loop. Rather than updating the values of the weights ourselves, we use an Optimizer object from the `torch.optim` package, which abstract the notion of an optimization algorithm and provides implementations of most of the algorithms commonly used to optimize neural networks.

### Module API：训练循环

我们同样会使用一个稍有不同的训练循环。这一次，我们不再手动更新权重的数值，而是使用 `torch.optim` 包中的 Optimizer 对象。Optimizer 抽象了优化算法的概念，并提供了大多数常用神经网络优化算法的实现。

In [21]:
def train_part34(model, optimizer, epochs=1):
    """
    Train a model on CIFAR-10 using the PyTorch Module API.

    Inputs:
    - model: A PyTorch Module giving the model to train.
    - optimizer: An Optimizer object we will use to train the model
    - epochs: (Optional) A Python integer giving the number of epochs to train for

    Returns: Nothing, but prints model accuracies during training.
    """
    model = model.to(device=device)  # move the model parameters to CPU/GPU
    for e in range(epochs):
        for t, (x, y) in enumerate(loader_train):
            model.train()  # put model to training mode
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            # Zero out all of the gradients for the variables which the optimizer
            # will update.
            optimizer.zero_grad()

            # This is the backwards pass: compute the gradient of the loss with
            # respect to each  parameter of the model.
            loss.backward()

            # Actually update the parameters of the model using the gradients
            # computed by the backwards pass.
            optimizer.step()

            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                check_accuracy_part34(loader_val, model)
                print()

### Module API: Train a Two-Layer Network
Now we are ready to run the training loop. In contrast to part II, we don't explicitly allocate parameter tensors anymore.

Simply pass the input size, hidden layer size, and number of classes (i.e. output size) to the constructor of `TwoLayerFC`.

You also need to define an optimizer that tracks all the learnable parameters inside `TwoLayerFC`.

You don't need to tune any hyperparameters, but you should see model accuracies above 40% after training for one epoch.

In [22]:
hidden_layer_size = 4000
learning_rate = 1e-2
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

train_part34(model, optimizer)

Iteration 0, loss = 3.3534
Checking accuracy on validation set
Got 168 / 1000 correct (16.80)

Iteration 100, loss = 2.8419
Checking accuracy on validation set
Got 297 / 1000 correct (29.70)

Iteration 200, loss = 1.9144
Checking accuracy on validation set
Got 404 / 1000 correct (40.40)

Iteration 300, loss = 2.4091
Checking accuracy on validation set
Got 384 / 1000 correct (38.40)

Iteration 400, loss = 1.6844
Checking accuracy on validation set
Got 381 / 1000 correct (38.10)

Iteration 500, loss = 1.5623
Checking accuracy on validation set
Got 438 / 1000 correct (43.80)

Iteration 600, loss = 1.9031
Checking accuracy on validation set
Got 436 / 1000 correct (43.60)

Iteration 700, loss = 1.5707
Checking accuracy on validation set
Got 436 / 1000 correct (43.60)



### Module API: Train a Three-Layer ConvNet
You should now use the Module API to train a three-layer ConvNet on CIFAR. This should look very similar to training the two-layer network! You don't need to tune any hyperparameters, but you should achieve above above 45% after training for one epoch.

You should train the model using stochastic gradient descent without momentum.
### Module API：训练一个三层卷积神经网络

现在你应该使用 Module API 在 CIFAR 数据集上训练一个三层卷积神经网络。这部分和训练两层网络非常相似！你不需要调整任何超参数，但训练一个 epoch 后准确率应该能超过 45%。

你应该使用不带动量的随机梯度下降（SGD）来训练模型。

In [23]:
learning_rate = 3e-3
channel_1 = 32
channel_2 = 16

model = None
optimizer = None
################################################################################
# TODO: Instantiate your ThreeLayerConvNet model and a corresponding optimizer #
################################################################################

model = ThreeLayerConvNet(in_channel=3, channel_1=channel_1, channel_2=channel_2, num_classes=10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)



################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

train_part34(model, optimizer)

Iteration 0, loss = 2.8912
Checking accuracy on validation set
Got 105 / 1000 correct (10.50)

Iteration 100, loss = 1.7797
Checking accuracy on validation set
Got 358 / 1000 correct (35.80)

Iteration 200, loss = 1.7045
Checking accuracy on validation set
Got 384 / 1000 correct (38.40)

Iteration 300, loss = 1.7736
Checking accuracy on validation set
Got 418 / 1000 correct (41.80)

Iteration 400, loss = 1.5553
Checking accuracy on validation set
Got 426 / 1000 correct (42.60)

Iteration 500, loss = 1.7453
Checking accuracy on validation set
Got 435 / 1000 correct (43.50)

Iteration 600, loss = 1.6611
Checking accuracy on validation set
Got 459 / 1000 correct (45.90)

Iteration 700, loss = 1.7540
Checking accuracy on validation set
Got 466 / 1000 correct (46.60)



# Part IV. PyTorch Sequential API

Part III introduced the PyTorch Module API, which allows you to define arbitrary learnable layers and their connectivity.

For simple models like a stack of feed forward layers, you still need to go through 3 steps: subclass `nn.Module`, assign layers to class attributes in `__init__`, and call each layer one by one in `forward()`. Is there a more convenient way?

Fortunately, PyTorch provides a container Module called `nn.Sequential`, which merges the above steps into one. It is not as flexible as `nn.Module`, because you cannot specify more complex topology than a feed-forward stack, but it's good enough for many use cases.

### Sequential API: Two-Layer Network
Let's see how to rewrite our two-layer fully connected network example with `nn.Sequential`, and train it using the training loop defined above.

Again, you don't need to tune any hyperparameters here, but you shoud achieve above 40% accuracy after one epoch of training.
# 第四部分：PyTorch Sequential API

第三部分介绍了 PyTorch 的 Module API，它允许你自定义任意可学习层及其连接方式。

对于像一堆前馈层这样简单的模型，你仍然需要经过 3 个步骤：继承 `nn.Module`，在 `__init__` 中将层赋值为类属性，并在 `forward()` 中逐个调用每一层。有没有更方便的方法？

幸运的是，PyTorch 提供了一个名为 `nn.Sequential` 的容器模块，它将上述步骤合并为一步。它没有 `nn.Module` 那么灵活，无法实现比前馈堆叠更复杂的结构，但对于很多场景已经足够用了。

### Sequential API：两层网络

让我们看看如何用 `nn.Sequential` 重写两层全连接网络的例子，并用上面定义的训练循环进行训练。

同样，这里你不需要调整任何超参数，但训练一个 epoch 后准确率应该能超过 40%。

In [24]:
# We need to wrap `flatten` function in a module in order to stack it
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 4000
learning_rate = 1e-2

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part34(model, optimizer)

Iteration 0, loss = 2.3047
Checking accuracy on validation set
Got 163 / 1000 correct (16.30)

Iteration 100, loss = 1.9230
Checking accuracy on validation set
Got 389 / 1000 correct (38.90)

Iteration 200, loss = 1.8187
Checking accuracy on validation set
Got 410 / 1000 correct (41.00)

Iteration 300, loss = 1.8021
Checking accuracy on validation set
Got 446 / 1000 correct (44.60)

Iteration 400, loss = 1.6752
Checking accuracy on validation set
Got 423 / 1000 correct (42.30)

Iteration 500, loss = 1.8070
Checking accuracy on validation set
Got 441 / 1000 correct (44.10)

Iteration 600, loss = 1.5292
Checking accuracy on validation set
Got 469 / 1000 correct (46.90)

Iteration 700, loss = 1.8148
Checking accuracy on validation set
Got 447 / 1000 correct (44.70)



### Sequential API: Three-Layer ConvNet
Here you should use `nn.Sequential` to define and train a three-layer ConvNet with the same architecture we used in Part III:

1. Convolutional layer (with bias) with 32 5x5 filters, with zero-padding of 2
2. ReLU
3. Convolutional layer (with bias) with 16 3x3 filters, with zero-padding of 1
4. ReLU
5. Fully-connected layer (with bias) to compute scores for 10 classes

You can use the default PyTorch weight initialization.

You should optimize your model using stochastic gradient descent with Nesterov momentum 0.9.

Again, you don't need to tune any hyperparameters but you should see accuracy above 55% after one epoch of training.
### Sequential API：三层卷积神经网络

这里你需要使用 `nn.Sequential` 来定义并训练一个三层卷积神经网络，结构与第三部分相同：

1. 卷积层（带偏置），使用 32 个 5x5 卷积核，零填充为 2  
2. ReLU 激活函数  
3. 卷积层（带偏置），使用 16 个 3x3 卷积核，零填充为 1  
4. ReLU 激活函数  
5. 全连接层（带偏置），用于计算 10 个类别的得分  

你可以使用 PyTorch 的默认权重初始化方式。

你应该使用带有 Nesterov 动量 0.9 的随机梯度下降（SGD）来优化模型。

同样，这里不需要调整任何超参数，但训练一个 epoch 后准确率应该能超过 55%。

In [25]:
channel_1 = 32
channel_2 = 16
learning_rate = 1e-2

model = None
optimizer = None

################################################################################
# TODO: Rewrite the 2-layer ConvNet with bias from Part III with the           #
# Sequential API.                                                              #
################################################################################
# We need to wrap `flatten` function in a module in order to stack it
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)



model = nn.Sequential(
    nn.Conv2d(3, channel_1, kernel_size=5, stride=1, padding=2),
    nn.ReLU(),
    nn.Conv2d(channel_1, channel_2, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    Flatten(),
    nn.Linear(channel_2 * 32 * 32, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)


################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

train_part34(model, optimizer)

Iteration 0, loss = 2.3184
Checking accuracy on validation set
Got 142 / 1000 correct (14.20)

Iteration 100, loss = 1.8258
Checking accuracy on validation set
Got 442 / 1000 correct (44.20)

Iteration 200, loss = 1.6102
Checking accuracy on validation set
Got 490 / 1000 correct (49.00)

Iteration 300, loss = 1.3024
Checking accuracy on validation set
Got 499 / 1000 correct (49.90)

Iteration 400, loss = 1.4486
Checking accuracy on validation set
Got 530 / 1000 correct (53.00)

Iteration 500, loss = 1.3054
Checking accuracy on validation set
Got 544 / 1000 correct (54.40)

Iteration 600, loss = 0.9590
Checking accuracy on validation set
Got 551 / 1000 correct (55.10)

Iteration 700, loss = 1.2708
Checking accuracy on validation set
Got 593 / 1000 correct (59.30)



# Part V. CIFAR-10 open-ended challenge

In this section, you can experiment with whatever ConvNet architecture you'd like on CIFAR-10.

Now it's your job to experiment with architectures, hyperparameters, loss functions, and optimizers to train a model that achieves **at least 70%** accuracy on the CIFAR-10 **validation** set within 10 epochs. You can use the check_accuracy and train functions from above. You can use either `nn.Module` or `nn.Sequential` API.

Describe what you did at the end of this notebook.

Here are the official API documentation for each component. One note: what we call in the class "spatial batch norm" is called "BatchNorm2D" in PyTorch.

* Layers in torch.nn package: http://pytorch.org/docs/stable/nn.html
* Activations: http://pytorch.org/docs/stable/nn.html#non-linear-activations
* Loss functions: http://pytorch.org/docs/stable/nn.html#loss-functions
* Optimizers: http://pytorch.org/docs/stable/optim.html


### Things you might try:
- **Filter size**: Above we used 5x5; would smaller filters be more efficient?
- **Number of filters**: Above we used 32 filters. Do more or fewer do better?
- **Pooling vs Strided Convolution**: Do you use max pooling or just stride convolutions?
- **Batch normalization**: Try adding spatial batch normalization after convolution layers and vanilla batch normalization after affine layers. Do your networks train faster?
- **Network architecture**: The network above has two layers of trainable parameters. Can you do better with a deep network? Good architectures to try include:
    - [conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]
    - [conv-relu-conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]
    - [batchnorm-relu-conv]xN -> [affine]xM -> [softmax or SVM]
- **Global Average Pooling**: Instead of flattening and then having multiple affine layers, perform convolutions until your image gets small (7x7 or so) and then perform an average pooling operation to get to a 1x1 image picture (1, 1 , Filter#), which is then reshaped into a (Filter#) vector. This is used in [Google's Inception Network](https://arxiv.org/abs/1512.00567) (See Table 1 for their architecture).
- **Regularization**: Add l2 weight regularization, or perhaps use Dropout.

### Tips for training
For each network architecture that you try, you should tune the learning rate and other hyperparameters. When doing this there are a couple important things to keep in mind:

- If the parameters are working well, you should see improvement within a few hundred iterations
- Remember the coarse-to-fine approach for hyperparameter tuning: start by testing a large range of hyperparameters for just a few training iterations to find the combinations of parameters that are working at all.
- Once you have found some sets of parameters that seem to work, search more finely around these parameters. You may need to train for more epochs.
- You should use the validation set for hyperparameter search, and save your test set for evaluating your architecture on the best parameters as selected by the validation set.

### Going above and beyond
If you are feeling adventurous there are many other features you can implement to try and improve your performance. You are **not required** to implement any of these, but don't miss the fun if you have time!

- Alternative optimizers: you can try Adam, Adagrad, RMSprop, etc.
- Alternative activation functions such as leaky ReLU, parametric ReLU, ELU, or MaxOut.
- Model ensembles
- Data augmentation
- New Architectures
  - [ResNets](https://arxiv.org/abs/1512.03385) where the input from the previous layer is added to the output.
  - [DenseNets](https://arxiv.org/abs/1608.06993) where inputs into previous layers are concatenated together.
  - [This blog has an in-depth overview](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)

### Have fun and happy training!
# Part V. CIFAR-10 open-ended challenge

In this section, you can experiment with whatever ConvNet architecture you'd like on CIFAR-10.

Now it's your job to experiment with architectures, hyperparameters, loss functions, and optimizers to train a model that achieves **at least 70%** accuracy on the CIFAR-10 **validation** set within 10 epochs. You can use the check_accuracy and train functions from above. You can use either `nn.Module` or `nn.Sequential` API.

Describe what you did at the end of this notebook.

Here are the official API documentation for each component. One note: what we call in the class "spatial batch norm" is called "BatchNorm2D" in PyTorch.

* Layers in torch.nn package: http://pytorch.org/docs/stable/nn.html
* Activations: http://pytorch.org/docs/stable/nn.html#non-linear-activations
* Loss functions: http://pytorch.org/docs/stable/nn.html#loss-functions
* Optimizers: http://pytorch.org/docs/stable/optim.html


### Things you might try:
- **Filter size**: Above we used 5x5; would smaller filters be more efficient?
- **Number of filters**: Above we used 32 filters. Do more or fewer do better?
- **Pooling vs Strided Convolution**: Do you use max pooling or just stride convolutions?
- **Batch normalization**: Try adding spatial batch normalization after convolution layers and vanilla batch normalization after affine layers. Do your networks train faster?
- **Network architecture**: The network above has two layers of trainable parameters. Can you do better with a deep network? Good architectures to try include:
    - [conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]
    - [conv-relu-conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]
    - [batchnorm-relu-conv]xN -> [affine]xM -> [softmax or SVM]
- **Global Average Pooling**: Instead of flattening and then having multiple affine layers, perform convolutions until your image gets small (7x7 or so) and then perform an average pooling operation to get to a 1x1 image picture (1, 1 , Filter#), which is then reshaped into a (Filter#) vector. This is used in [Google's Inception Network](https://arxiv.org/abs/1512.00567) (See Table 1 for their architecture).
- **Regularization**: Add l2 weight regularization, or perhaps use Dropout.

### Tips for training
For each network architecture that you try, you should tune the learning rate and other hyperparameters. When doing this there are a couple important things to keep in mind:

- If the parameters are working well, you should see improvement within a few hundred iterations
- Remember the coarse-to-fine approach for hyperparameter tuning: start by testing a large range of hyperparameters for just a few training iterations to find the combinations of parameters that are working at all.
- Once you have found some sets of parameters that seem to work, search more finely around these parameters. You may need to train for more epochs.
- You should use the validation set for hyperparameter search, and save your test set for evaluating your architecture on the best parameters as selected by the validation set.

### Going above and beyond
If you are feeling adventurous there are many other features you can implement to try and improve your performance. You are **not required** to implement any of these, but don't miss the fun if you have time!

- Alternative optimizers: you can try Adam, Adagrad, RMSprop, etc.
- Alternative activation functions such as leaky ReLU, parametric ReLU, ELU, or MaxOut.
- Model ensembles
- Data augmentation
- New Architectures
  - [ResNets](https://arxiv.org/abs/1512.03385) where the input from the previous layer is added to the output.
  - [DenseNets](https://arxiv.org/abs/1608.06993) where inputs into previous layers are concatenated together.
  - [This blog has an in-depth overview](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)

### Have fun and happy training!
# 第五部分：CIFAR-10 开放挑战

在本节中，你可以在 CIFAR-10 上尝试任何你喜欢的卷积神经网络结构。

现在轮到你来尝试不同的网络结构、超参数、损失函数和优化器，训练一个模型，使其在 10 个 epoch 内在 CIFAR-10 **验证集**上达到**至少 70%** 的准确率。你可以使用上面提供的 check_accuracy 和 train 函数。可以选择 `nn.Module` 或 `nn.Sequential` API。

请在本 Notebook 结尾处描述你的实验过程和结果。

以下是各组件的官方 API 文档。注意：课程中所说的“spatial batch norm”在 PyTorch 中叫做“BatchNorm2D”。

* torch.nn 包中的层: http://pytorch.org/docs/stable/nn.html
* 激活函数: http://pytorch.org/docs/stable/nn.html#non-linear-activations
* 损失函数: http://pytorch.org/docs/stable/nn.html#loss-functions
* 优化器: http://pytorch.org/docs/stable/optim.html

---

### 你可以尝试的内容：
- **卷积核大小**：上面我们用的是 5x5，更小的卷积核会更高效吗？
- **卷积核数量**：上面我们用了 32 个卷积核，更多或更少会更好吗？
- **池化 vs 步长卷积**：你会用最大池化还是只用步长卷积？
- **批归一化**：尝试在卷积层后加空间批归一化，在全连接层后加普通批归一化。你的网络会训练得更快吗？
- **网络结构**：上面的网络只有两层可训练参数。更深的网络会更好吗？可以尝试的结构包括：
    - [conv-relu-pool]xN -> [affine]xM -> [softmax 或 SVM]
    - [conv-relu-conv-relu-pool]xN -> [affine]xM -> [softmax 或 SVM]
    - [batchnorm-relu-conv]xN -> [affine]xM -> [softmax 或 SVM]
- **全局平均池化**：不再展平后接多层全连接，而是卷积到图片变小（如 7x7），再做一次平均池化变成 1x1（1, 1, Filter#），最后 reshape 成 (Filter#) 向量。这在 [Google 的 Inception 网络](https://arxiv.org/abs/1512.00567) 中有用（见 Table 1）。
- **正则化**：加 l2 权重正则化，或使用 Dropout。

---

### 训练小贴士
对于每个你尝试的网络结构，都应该调节学习率和其他超参数。调参时有几点很重要：

- 如果参数设置得好，几百次迭代内应该能看到提升
- 记住粗到细的调参方法：先用较大范围的超参数做少量训练，找到能收敛的参数组合；
- 找到有效参数后，再在这些参数附近细致搜索，可能需要训练更多 epoch。
- 用验证集做超参数搜索，测试集只用来在最终最优参数下评估架构。

---

### 更进一步
如果你有兴趣，还可以尝试以下高级特性（非必做，但很有趣）：

- 其他优化器：如 Adam、Adagrad、RMSprop 等
- 其他激活函数：如 leaky ReLU、parametric ReLU、ELU 或 MaxOut
- 模型集成
- 数据增强
- 新的网络结构
  - [ResNets](https://arxiv.org/abs/1512.03385)：前一层的输入加到输出上
  - [DenseNets](https://arxiv.org/abs/1608.06993)：前面所有层的输入拼接到一起
  - [这篇博客有详细介绍](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)

---

### 祝你玩得开心，训练顺利！

In [26]:
################################################################################
# TODO:                                                                        #
# Experiment with any architectures, optimizers, and hyperparameters.          #
# Achieve AT LEAST 70% accuracy on the *validation set* within 10 epochs.      #
#                                                                              #
# Note that you can use the check_accuracy function to evaluate on either      #
# the test set or the validation set, by passing either loader_test or         #
# loader_val as the second argument to check_accuracy. You should not touch    #
# the test set until you have finished your architecture and  hyperparameter   #
# tuning, and only run the test set once at the end to report a final value.   #
################################################################################

class CustomNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)      # 3x32x32 -> 64x32x32
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)    # 64x32x32 -> 128x32x32
        self.bn2 = nn.BatchNorm2d(128)
        self.pool1 = nn.MaxPool2d(2, 2)                              # 128x32x32 -> 128x16x16

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)   # 128x16x16 -> 256x16x16
        self.bn3 = nn.BatchNorm2d(256)
        self.pool2 = nn.MaxPool2d(2, 2)                              # 256x16x16 -> 256x8x8

        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256*8*8, 512)
        self.bn_fc = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.fc2(x)
        return x

model = CustomNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

# You should get at least 70% accuracy
train_part34(model, optimizer, epochs=10)

Iteration 0, loss = 2.3550
Checking accuracy on validation set
Got 106 / 1000 correct (10.60)

Iteration 100, loss = 1.4696
Checking accuracy on validation set
Got 531 / 1000 correct (53.10)

Iteration 200, loss = 1.1463
Checking accuracy on validation set
Got 607 / 1000 correct (60.70)

Iteration 300, loss = 0.9365
Checking accuracy on validation set
Got 618 / 1000 correct (61.80)

Iteration 400, loss = 0.9266
Checking accuracy on validation set
Got 667 / 1000 correct (66.70)

Iteration 500, loss = 0.7914
Checking accuracy on validation set
Got 682 / 1000 correct (68.20)

Iteration 600, loss = 0.8211
Checking accuracy on validation set
Got 700 / 1000 correct (70.00)

Iteration 700, loss = 0.7402
Checking accuracy on validation set
Got 696 / 1000 correct (69.60)

Iteration 0, loss = 0.8617
Checking accuracy on validation set
Got 689 / 1000 correct (68.90)

Iteration 100, loss = 0.6200
Checking accuracy on validation set
Got 717 / 1000 correct (71.70)

Iteration 200, loss = 0.6514
Check

## Describe what you did

In the cell below you should write an explanation of what you did, any additional features that you implemented, and/or any graphs that you made in the process of training and evaluating your network.

**Answer:**



## Test set -- run this only once

Now that we've gotten a result we're happy with, we test our final model on the test set (which you should store in best_model). Think about how this compares to your validation set accuracy.

In [27]:
best_model = model
check_accuracy_part34(loader_test, best_model)

Checking accuracy on test set
Got 8232 / 10000 correct (82.32)
